# Test 3 Option B: Speech-to-Speech (LMGen with float32 fix)
Prerequisite: Run moshiko2.0.ipynb first. This reloads models with float32 to fix T4 bfloat16 error.

In [ ]:
# Step 1: Reload models with float32 fix for T4
import torch, time, os
from moshi.models import loaders, LMGen

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Disable bfloat16 autocast - T4 doesn't support it natively
torch.set_autocast_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("Autocast and TF32 disabled.")

# Locate model files
Q8_DIR = FOLDERS["models_q8"]
mimi_path = os.path.join(Q8_DIR, "tokenizer-e351c8d8-checkpoint125.safetensors")
moshi_path = os.path.join(Q8_DIR, "model.q8.safetensors")

# Load Mimi
print("\nLoading Mimi (float32)...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.float()
mimi.eval()
print(f"  Mimi loaded in {time.time()-start:.1f}s")

# Load Moshiko LM
print("\nLoading Moshiko LM (float32, quantize=True)...")
start = time.time()

lm_kwargs = {
    "delays": [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    "n_q": 16,
    "dep_q": 8,
    "card": 2048,
    "text_card": 32000,
    "existing_text_padding_id": 3,
    "dim": 4096,
    "num_heads": 32,
    "num_layers": 32,
    "hidden_scale": 4.125,
    "causal": True,
    "context": 3000,
    "max_period": 10000,
    "gating": "silu",
    "norm": "rms_norm_f32",
    "positional_embedding": "rope",
    "layer_scale": None,
    "depformer_dim": 1024,
    "depformer_dim_feedforward": 4224,
    "depformer_num_heads": 16,
    "depformer_num_layers": 6,
    "depformer_causal": True,
    "depformer_layer_scale": None,
    "depformer_multi_linear": True,
    "depformer_context": 8,
    "depformer_max_period": 10000,
    "depformer_gating": "silu",
    "depformer_pos_emb": "none",
    "depformer_weights_per_step": True,
    "quantize": True,
}

moshi_lm = loaders.get_moshi_lm(moshi_path, device=DEVICE, lm_kwargs=lm_kwargs)
moshi_lm.float()  # Force float32 - prevents bfloat16 cast on weight_scb
moshi_lm.eval()
print(f"  Moshiko LM loaded in {time.time()-start:.1f}s")

# Create LMGen
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  LMGen ready.")

torch.cuda.empty_cache()
if DEVICE == "cuda":
    print(f"  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Store globally
import builtins
builtins.mimi = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen = lm_gen
builtins.DEVICE = DEVICE

print("\nModels reloaded with float32. Ready for generation.")

In [ ]:
# Step 2: Speech-to-speech generation
import torch, soundfile as sf
import numpy as np, time
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000
MAX_RESPONSE_FRAMES = 200

# Use input audio (run Test 1 first, or provide your own speech WAV)
input_path = f"{FOLDERS['audio_in']}/test_input_440hz.wav"
if not os.path.exists(input_path):
    print(f"Input file not found: {input_path}")
    print("Run Test 1 first, or upload a speech WAV and update input_path.")
else:
    wav_np, sr = sf.read(input_path)
    wav = torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    print(f"Input: {input_path} | Duration: {wav.shape[-1]/SAMPLE_RATE:.2f}s")

    # Encode
    with torch.no_grad():
        input_codes = mimi.encode(wav)
    print(f"Input codes shape: {input_codes.shape}")

    # Generate response - create fresh LMGen for each generation
    print("\nGenerating response...")
    gen_start = time.time()

    lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
    response_codes = []

    with torch.no_grad():
        num_input_frames = input_codes.shape[-1]
        for i in range(num_input_frames):
            frame = input_codes[:, :, i:i+1]
            lm_gen.step(frame)

        for i in range(MAX_RESPONSE_FRAMES):
            out_tokens = lm_gen.step(None)
            if out_tokens is not None:
                response_codes.append(out_tokens)

    gen_time = time.time() - gen_start
    print(f"Generated {len(response_codes)} frames in {gen_time:.1f}s")

    if response_codes:
        response_tensor = torch.cat(response_codes, dim=-1)
        with torch.no_grad():
            response_audio = mimi.decode(response_tensor)

        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_path = f"{FOLDERS['audio_out']}/test3b_response_{ts}.wav"
        sf.write(output_path, response_audio.squeeze().cpu().numpy(), SAMPLE_RATE)

        print(f"Response duration: {response_audio.shape[-1]/SAMPLE_RATE:.2f}s")
        print(f"Generation speed: {gen_time/len(response_codes)*1000:.0f} ms/frame")

        print("\nInput audio:")
        display(Audio(wav_np, rate=SAMPLE_RATE))
        print("Response audio:")
        display(Audio(response_audio.squeeze().cpu().numpy(), rate=SAMPLE_RATE))

        result = {
            "test": "Speech-to-speech (Option B: LMGen float32)",
            "timestamp": ts,
            "input_file": input_path,
            "output_file": output_path,
            "input_frames": num_input_frames,
            "response_frames": len(response_codes),
            "response_duration_s": round(response_audio.shape[-1]/SAMPLE_RATE, 2),
            "gen_time_s": round(gen_time, 2),
            "ms_per_frame": round(gen_time/len(response_codes)*1000, 1),
        }
        import json
        with open(f"{FOLDERS['outputs']}/test3b_results_{ts}.json", "w") as f:
            json.dump(result, f, indent=2)

        print(f"\nTest 3 Option B complete. Files saved to Drive.")
    else:
        print("No response generated. Try with real speech audio, not synthetic tones.")